# <center>C6: Deep Learning Optimization Strategies</center>

## 1. Introduction to Optimization

Optimization is the heart of Deep Learning. It is the process of adjusting the weights of a neural network to minimize the **Cost Function** (or Loss Function).

### The Loss Landscape
Imagine the Cost Function as a landscape of hills and valleys. Our goal is to reach the lowest point (Global Minimum).

-   **Convex Functions**: Like a bowl. There is only one minimum, and it is easy to find.
-   **Non-Convex Functions**: Like a rugged mountain range. There are many "Local Minima" (shallow valleys) and "Saddle Points" where the gradient is zero but it's not the bottom.

![Convex vs Non-Convex Landscape](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.engineeringdevotion.com%2Fimg%2Foptim%2Fclass-5.PNG&f=1&nofb=1&ipt=d88642dbec82b88a1dcf72047df6aaa023e6bbda7b0f1d4da99e98a05ef2691d)

### Gradient Descent Core
The primary algorithm we use is **Gradient Descent**.
1.  Calculate the gradient (slope) of the loss with respect to weights: $\nabla J(W)$.
2.  Update weight in the opposite direction of the gradient:
    $$W_{new} = W_{old} - \alpha \times \nabla J(W)$$
    where $\alpha$ is the **Learning Rate**.

## 2. Variants of Gradient Descent

Depending on how much data we use to calculate the gradient, we have three main variants:

### 2.1 Batch Gradient Descent
-   Uses the **Entire Dataset** to compute the gradient.
-   **Pros**: Stable, smooth convergence graph.
-   **Cons**: Very slow for large datasets; requires huge memory.

### 2.2 Stochastic Gradient Descent (SGD)
-   Uses **One Single Sample** to compute the gradient.
-   **Pros**: Very fast updates; noise can help escape local minima.
-   **Cons**: Extremely noisy convergence (zig-zag path); never settles completely.

### 2.3 Mini-Batch Gradient Descent
-   Uses a small **Batch** (e.g., 32, 64, 128 samples).
-   **The Gold Standard**: Balances the stability of Batch GD with the speed of SGD.

![GD Variants Comparison](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.nomidl.com%2Fwp-content%2Fuploads%2F2022%2F08%2Fimage-11.png&f=1&nofb=1&ipt=2406c4a7b0fa2076acf6790a3a907848105deba335c8bef033965e85bd2089c9)
*Visualizing the path of descent: Batch (smooth), Stochastic (noisy), Mini-batch (balanced).*

## 3. Advanced Optimizers

Standard SGD has trouble navigating ravines (areas where the surface curves much more steeply in one dimension than another). Modern optimizers fix this.

### 3.1 Momentum
-   **Concept**: Accumulates a velocity vector in directions of persistent reduction in the objective across iterations.
-   **Analogy**: A ball rolling down a hill gains momentum, making it faster and harder to change direction due to small bumps.

### 3.2 RMSprop (Root Mean Square Propagation)
-   **Concept**: Adaptive learning rate. Dividing the gradient by a running average of its recent magnitude.
-   **Effect**: Slows down learning in steep vertical directions (large gradients) and speeds up in flat horizontal directions.

### 3.3 Adam (Adaptive Moment Estimation)
-   **Concept**: Combines **Momentum** (1st moment) and **RMSprop** (2nd moment).
-   **Verdict**: The default choice for most problems today.

## 4. Learning Rate Scheduling

Choosing the right learning rate is crucial. A static learning rate is often not optimal.

![Learning Rate Effect](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.jeremyjordan.me%2Fcontent%2Fimages%2F2018%2F02%2FScreen-Shot-2018-02-24-at-11.47.09-AM.png&f=1&nofb=1&ipt=a7be1c0d8a5fa98dc52f70ba7f75d5932ddbb43808dbbfb2a9216d4b5d5878d5)

-   **Too Large**: Diverges or oscillates.
-   **Too Small**: Takes forever to converge.
-   **Scheduling**: Start high (to travel fast), then decay (to settle in the minimum).

![Learning Rate Scheduling](https://external-content.duckduckgo.com/iu/?u=http%3A%2F%2Fwww.bdhammel.com%2Fassets%2Flearning-rate%2Flr-types.png&f=1&nofb=1&ipt=515b5399426fe2b914b382618f1f385616413ee208a9d4429ad98e5d3a4a75d5)
### Implementation: ReduceLROnPlateau
A popular Keras callback that reduces LR when a metric has stopped improving.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from tensorflow.keras.callbacks import ReduceLROnPlateau

# Define the Callback
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',   # Watch validation loss
    factor=0.5,           # Reduce LR by half
    patience=5,           # Wait 5 epochs with no improvement
    min_lr=0.00001        # Lower bound
)

# Usage in model.fit:
# history = model.fit(..., callbacks=[lr_scheduler])

## 5. Batch Normalization

Deep networks suffer from **Internal Covariate Shift**: The distribution of inputs to deep layers changes as weights in previous layers change.

![Batch Normalization](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmiro.medium.com%2Fv2%2Fresize%3Afit%3A898%2F0*-SwM1XOPnrx2OKxN.png&f=1&nofb=1&ipt=b3b1438cabb4d1b1d11b4d5e7796045190693ec298c854d0fb755763ef8987a3)

**Batch Normalization** fixes this by normalizing layer inputs (mean=0, variance=1) for every batch. This:
1.  Allows higher learning rates.
2.  Acts as a regularizer (like Dropout).
3.  Accelerates training significantly.

In [ ]:
from tensorflow.keras.layers import BatchNormalization

model_bn = Sequential([
    Input(shape=(2,)),  
    Dense(64),
    BatchNormalization(), # Add after layer, before activation (usually)
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dense(1, activation='sigmoid')
])

model_bn.summary()

## 7. Hands-On Optimizer Comparison

Now that you understand how different optimizers work in theory, let's see them in action! We'll train the same model with three different optimizers and compare their performance.

This hands-on comparison will help you understand:
- How SGD behaves vs modern adaptive optimizers
- The impact of momentum on training
- Why Adam is often the default choice

Let's build a simple classification task and race these optimizers against each other!

In [ ]:
# 1. Create synthetic non-linear data
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt

X, y = make_moons(n_samples=1000, noise=0.1, random_state=42)

# 2. Function to build and train a model with different optimizers
def train_with_optimizer(optimizer, optimizer_name, epochs=50):
    """Train a model with a specific optimizer and return training history"""
    model = Sequential([
        Input(shape=(2,)),
        Dense(50, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(loss='binary_crossentropy', 
                  optimizer=optimizer,
                  metrics=['accuracy'])
    
    history = model.fit(X, y, 
                       validation_split=0.3,
                       epochs=epochs,
                       verbose=0,  # Silent training
                       batch_size=32)
    
    return history

# 3. Train with three different optimizers
print("Training with different optimizers...")
print("=" * 50)

history_sgd = train_with_optimizer(
    SGD(learning_rate=0.01, momentum=0.0), 
    'SGD'
)
print("✓ SGD (no momentum) completed")

history_sgd_momentum = train_with_optimizer(
    SGD(learning_rate=0.01, momentum=0.9),
    'SGD + Momentum'
)
print("✓ SGD + Momentum completed")

history_adam = train_with_optimizer(
    Adam(learning_rate=0.01),
    'Adam'
)
print("✓ Adam completed")

# 4. Visualize the comparison
plt.figure(figsize=(14, 5))

# Plot Training Loss
plt.subplot(1, 2, 1)
plt.plot(history_sgd.history['loss'], label='SGD', linewidth=2)
plt.plot(history_sgd_momentum.history['loss'], label='SGD + Momentum', linewidth=2)
plt.plot(history_adam.history['loss'], label='Adam', linewidth=2)
plt.title('Training Loss Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot Validation Accuracy
plt.subplot(1, 2, 2)
plt.plot(history_sgd.history['val_accuracy'], label='SGD', linewidth=2)
plt.plot(history_sgd_momentum.history['val_accuracy'], label='SGD + Momentum', linewidth=2)
plt.plot(history_adam.history['val_accuracy'], label='Adam', linewidth=2)
plt.title('Validation Accuracy Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. Final comparison
print("\n" + "=" * 50)
print("FINAL RESULTS (Epoch 50):")
print("=" * 50)
print(f"SGD               - Loss: {history_sgd.history['loss'][-1]:.4f}, Val Acc: {history_sgd.history['val_accuracy'][-1]:.4f}")
print(f"SGD + Momentum    - Loss: {history_sgd_momentum.history['loss'][-1]:.4f}, Val Acc: {history_sgd_momentum.history['val_accuracy'][-1]:.4f}")
print(f"Adam              - Loss: {history_adam.history['loss'][-1]:.4f}, Val Acc: {history_adam.history['val_accuracy'][-1]:.4f}")

### Analysis of Results

**What did we observe?**

1. **SGD (no momentum)**: 
   - Slowest convergence
   - More noisy training curve
   - Can get stuck in local minima or saddle points

2. **SGD + Momentum**:
   - Faster than plain SGD
   - Smoother training curve
   - Momentum helps escape shallow local minima

3. **Adam**:
   - Fastest convergence
   - Most stable training
   - Adaptive learning rates per parameter
   - **Winner** for most practical applications!

**Key Takeaway**: This is why Adam is the default choice in modern deep learning. It combines the best of both worlds: momentum + adaptive learning rates.

## 8. Exercises

**Exercise 1: Optimizer on Your Own Dataset**
Try the optimizer comparison from Section 7 on the `UCI Credit Card Data.csv` dataset. Which optimizer performs best? Does the ranking change with a different dataset?

**Exercise 2: Learning Rate Experimentation**
Modify the Adam optimizer in Section 7 to use different learning rates: 0.001, 0.01, 0.1. How does this affect convergence speed and final accuracy?

**Exercise 3: Effect of Batch Normalization**
Build two identical models (using Adam):
1. Model A: Without Batch Normalization.
2. Model B: With Batch Normalization.
Train them on the moon dataset from Section 7 and compare training curves. Does Batch Norm help?

In [ ]:
# Load UCI Credit Card Data
df = pd.read_csv('../../00-data/UCI Credit Card Data.csv')
df.head()